In [ ]:
!pip install -q google-generativeai python-dotenv


In [ ]:
# Set your API key
import os
os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"
print(os.getenv("GOOGLE_API_KEY"))


GOOGLE_API_KEY


In [ ]:
import json
import time
from dotenv import load_dotenv
import google.generativeai as genai


In [ ]:
# --- Configuration --- #
def load_environment():
    """Load environment variables from .env file"""
    load_dotenv('./gemini.env')  # Create this file with GEMINI_API_KEY=your_key
    return os.getenv("GEMINI_API_KEY")


In [ ]:
# --- Text Processing --- #
def create_correction_prompt(text: str) -> str:
    """Create structured prompt for text correction"""
    return f"""
    Correct the following historical Spanish text while PRESERVING ORIGINAL GRAMMAR AND STYLE.
    Only fix orthographic errors and punctuation. Maintain original capitalization and formatting.
    Return ONLY the corrected text without additional comments.

    Input text:
    {text}

    Corrected text:
    """


In [ ]:
def process_text(text: str, max_retries: int = 3) -> tuple:
    """Process text through Gemini API with error handling"""
    prompt = create_correction_prompt(text)

    for attempt in range(max_retries):
        try:
            response = genai.GenerativeModel('gemini-2.5-pro-exp-03-25').generate_content(prompt)
            if response.candidates and response.text:
                return response.text.strip(), "success"
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            time.sleep(2 ** attempt)  # Exponential backoff

    return "", "max_retries_exceeded"


In [ ]:
# --- Main Execution --- #
def main():
    # Initialize API
    api_key = load_environment()
    genai.configure(api_key=api_key)

    # Input text (replace with your text)
    input_text = '''Y pues  en la  celebrial
pues en la  celebrial Jeru-

falén no ha mudado de condicion
amidad    «maña»
vueftra  Benignidad,  profeguid
ignidad,  proleguid,
(1,1)
o  Niño  tierno, y  Dios Eterno,
, Y «DIOS CIETNO,
id en bendeci
rofeguid en bendecirles, y favo-
Ies
profeguid en bendecirles, y
recerles.  Sean tan fervorofamen

te devotos de vueftra  Admirable
MADRE, que le porten como fus
, que le por
hijos, y hermanos de leche  con

Vos.  Serán fabios, fi fueren caf-
.  Seran labios y

tos ; que no entra vueltra  Sabi-

duría, donde no ay mucha pure-
, donde no ay mucha p
"an a"
za de  conciencia.   Crezcan en

vueftro lanto temór, y amor, co-
, y amor,

como en los años, y mucho  más
,y mucho  más.

Adelantente en la virtud,  como
en las letras, y mucho  más ; haf-
, y mucho  más ;
.
ta que lleguen,  por vueltra imi-
que lleguen,  por
ion, à  fer varones perfect
tacion, a fer varones perfectos,
fumados,    agradables   à
y  confumados    agradables
vueftros ojos, y provechofos à
jos, y p
“,
la República,  que libra café-
publica,  q
.
da  fu felizidad en  la  acerrada

y    «  » ,   crían:

'''

    # Process text
    start_time = time.time()
    corrected_text, status = process_text(input_text)
    processing_time = time.time() - start_time

    # Save results
    result = {
        "metadata": {
            "model": "gemini-2.5-pro-exp-03-25",
            "status": status,
            "processing_time": round(processing_time, 2),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        },
        "original_text": input_text,
        "corrected_text": corrected_text
    }

    with open("text_correction_results.json", "w", encoding="latin",errors="ignore") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    # Display results
    print("═" * 50)
    print("Original text:\n")
    print(input_text)
    print("\n" + "═" * 50)
    print("Corrected text:\n")
    print(corrected_text)
    print("\n" + "═" * 50)
    print(f"Status: {status} | Processing time: {processing_time:.2f}s")

if __name__ == "__main__":
    main()

══════════════════════════════════════════════════
Original text:

Y pues  en la  celebrial
pues en la  celebrial Jeru-

falén no ha mudado de condicion
amidad    «maña»
vueftra  Benignidad,  profeguid
ignidad,  proleguid,
(1,1)
o  Niño  tierno, y  Dios Eterno,
, Y «DIOS CIETNO,
id en bendeci
rofeguid en bendecirles, y favo-
Ies
profeguid en bendecirles, y
recerles.  Sean tan fervorofamen

te devotos de vueftra  Admirable
MADRE, que le porten como fus
, que le por
hijos, y hermanos de leche  con

Vos.  Serán fabios, fi fueren caf-
.  Seran labios y

tos ; que no entra vueltra  Sabi-

duría, donde no ay mucha pure-
, donde no ay mucha p
"an a"
za de  conciencia.   Crezcan en

vueftro lanto temór, y amor, co-
, y amor,

como en los años, y mucho  más
,y mucho  más.

Adelantente en la virtud,  como
en las letras, y mucho  más ; haf-
, y mucho  más ;
.
ta que lleguen,  por vueltra imi-
que lleguen,  por
ion, à  fer varones perfect
tacion, a fer varones perfectos,
fumados,    agradables   à

In [ ]:
import editdistance
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def calculate_cer(text1, text2):
    """Calculate Character Error Rate (CER) between two texts."""
    text1 = text1.replace(" ", "").replace("\n", "")
    text2 = text2.replace(" ", "").replace("\n", "")

    # Calculate edit distance and CER
    distance = editdistance.eval(text1, text2)
    cer = distance / max(len(text1), len(text2))

    return cer


def calculate_wer(text1, text2):
    """Calculate Word Error Rate (WER) between two texts."""
    words1 = text1.split()
    words2 = text2.split()

    # Calculate edit distance and WER
    distance = editdistance.eval(words1, words2)
    wer = distance / max(len(words1), len(words2))

    return wer


def calculate_cosine_similarity(text1, text2):
    """Calculate Cosine Similarity between two texts."""
    vectorizer = CountVectorizer().fit_transform([text1, text2])
    vectors = vectorizer.toarray()
    cos_sim = np.dot(vectors[0], vectors[1]) / (np.linalg.norm(vectors[0]) * np.linalg.norm(vectors[1]))
    return cos_sim


def calculate_bleu(text1, text2):
    """Calculate BLEU score between two texts."""
    reference = [text2.split()]
    candidate = text1.split()
    smoothing_function = SmoothingFunction().method1
    return sentence_bleu(reference, candidate, smoothing_function=smoothing_function)

# text1 = '''el dedo quarto de la mano: luego se le ve el modo
# dos dedos pulgar y índice, doblando cada letra en su
# lugar, poniendo mucho cuidado en no errarlos, pero tener
# demás tripas y
# espacios poco que corregir. La letra, pues que es tema sabido,
# se distribuya con más velocidad, ha de estar
# le abrocha el componedor con los quatro dedos,
# palma de la mano izquierda y el dedo pulgar de
# la mano derecha trae, y la acomoda como va en est
# El componedor, ha de estar la vist
# a do la letra viene del de la caxa al componedor, ha
# que le ha de tomar, para tomarla por la ca
# letra siguiente que le ha de tomar, para tomarla
# beca por la mayor brevedad: porque como este
# quiera cosa que se le aprevie en cada uno.
# m de tantos tiempos, cualquiera cosa que
# principio le ha de ir
# hace muchos al fin del día Cierto es que al
# se va habituando: más en
# espacio, hasta que las manos se vayan habitu
# ando habituadas, se ha de procurar que con la mano dere
# cha traiga la letra con serenidad, y reposo, sin hacer
# sonecícos, ni
# movimientos no necesarios, de que algunos tienen grande vi-
# cio, que cuando quieren no lo pueden remediar, y no sirven sino
# de gastar el tiempo sin
# fruto. En teniendo ya las manos sueltas, y
# hechas á estos movimientos, se verá con experiencia
# más que luce la composición.'''
# # Example usage
# text2 = '''en el dedo quarto de la mano: luego en lyeyendo lo que ha toma-
# do, ir con los dos dedos pulgar, y indice echando cada etra en su
# caxoncito, poniendo mucho cuidado en no errarlos, para tener
# después poco que corregir. La letra, para que este más tratable, y
# se distribuya con más velocidad, ha de estar siempre mojada.
# Para componer se abraza el componedor con los quatro dedos,
# y palma de la mano izquierda, y el dedo pulgar recibe la letra que
# la mano derecha trae, y la acomoda como ha de estar: y en qua-
# to la letra viene desde la caxa al componedor, ha de estar la vista
# en la letra siguiente te que se ha de tomar, para tomarla por la ca-
# beza, por la mayor brevedad: porque como este exercicio cons
# ta de tanto tiempos, qualquiera cosa que se abrevie en cada vno
# haze mucho al fin del dia. Cierto es, que al principio se ha de ir
# con espacio, hasta que las manos se vayan habituando: mas en
# estando habituadas, se hace procurar que con la mano derecha
# se traiga la letra con serenidad, y reposo, sin hacer sonecitos, ni
# movimientos no necessarios, de que algunos tienen grande vi-
# cio, que quando quieren no lo pueden remediar, y no serven sino
# de gastar el tiempo sin fruto. En teniendo ya las manos sueltas, y
# hechas á estos movimientos, se verá con experiencia lo mucho
# mas que luce la composicion.'''
text1 = '''Y pues  en la  celestial Jeru-
salén no ha mudado de condición
vuestra  Benignidad,  proseguid
o  Niño  tierno, y  Dios Eterno,
proseguid en bendecirles, y favo-
recerles.  Sean tan fervorosamente
te devotos de vuestra  Admirable
MADRE, que le porten como sus
hijos, y hermanos de leche  con
Vos.  Serán sabios, si fueren cas-
tos ; que no entra vuestra  Sabi-
duría, donde no hay mucha pure-
za de  conciencia.   Crezcan en
vuestro santo temor, y amor, co-
mo en los años, y mucho  más.
Adelanten en la virtud,  como
en las letras, y mucho  más ; has-
ta que lleguen,  por vuestra imi-
tación, a ser varones perfectos,
y consumados,    agradables   a
vuestros ojos, y provechosos a
la República,  que confia-
da su felicidad en  la  acertada
cría:'''
text2 = '''Y pues en la celestial Jeru- salen no ha mudado de condicion vuestra Benignidad, proseguid, o Niño tierno, y Dios Eterno, proseguid en bendecirles, y favo- recerles. Sean tan fervorosamen- te devotos de vuestra Admirable MADRE, que se porten como sus hijos, y hermanos de leche con Vos. Seran sabios, si fueren cas- tos; que no entra vuestra Sabi- duria, donde no ay mucha pure- za de conciencia. Crezcan en vuestro santo temor, y amor, co- como en los años, y mucho mas. Adelantense en la virtud, como en las letras, y mucho mas; has- ta que lleguen, por vuesetra imi- tacion, a ser varones perfectos, y consumados, agradables a vuestros ojos, y provechosos a la Republica, que libra casi to- da su felizidad en la acertada'''
print(f"CER: {calculate_cer(text1, text2):.4f}")
print(f"WER: {calculate_wer(text1, text2):.4f}")
print(f"Cosine Similarity: {calculate_cosine_similarity(text1, text2):.4f}")
print(f"BLEU Score: {calculate_bleu(text1, text2):.4f}")

def calculate_jaccard_similarity(text1, text2):
    """Calculate Jaccard Similarity between two texts."""
    set1 = set(text1.split())
    set2 = set(text2.split())
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union != 0 else 0.0


print(f"Jaccard Similarity: {calculate_jaccard_similarity(text1, text2):.4f}")


CER: 0.0569
WER: 0.1797
Cosine Similarity: 0.9099
BLEU Score: 0.6353
Jaccard Similarity: 0.6754


In [ ]:
backslash_char = "\n"

In [ ]:
import difflib
import re
from nltk.tokenize import WordPunctTokenizer
from dataclasses import dataclass


In [ ]:
# Install NLTK punkt if needed
try:
    WordPunctTokenizer().tokenize("test")
except LookupError:
    import nltk
    nltk.download('punkt')


In [ ]:
@dataclass
class TextChange:
    original: str
    corrected: str
    start: int
    end: int
    context: str
    is_ocr_error: bool

def get_text_changes(original: str, corrected: str) -> list[TextChange]:
    """Identify text changes between original and corrected text"""
    tokenizer = WordPunctTokenizer()

    # Get tokens with spans
    orig_tokens = tokenizer.tokenize(original)
    orig_spans = list(tokenizer.span_tokenize(original))

    corr_tokens = tokenizer.tokenize(corrected)
    corr_spans = list(tokenizer.span_tokenize(corrected))

    sm = difflib.SequenceMatcher(None, orig_tokens, corr_tokens)
    changes = []

    for opcode, a0, a1, b0, b1 in sm.get_opcodes():
        if opcode != 'replace':
            continue

        # Get original text span
        orig_start = orig_spans[a0][0]
        orig_end = orig_spans[a1-1][1] if a1 > 0 else 0
        orig_text = original[orig_start:orig_end]

        # Get corrected text span
        corr_start = corr_spans[b0][0]
        corr_end = corr_spans[b1-1][1] if b1 > 0 else 0
        corr_text = corrected[corr_start:corr_end]

        # OCR error detection
        clean_orig = re.sub(r'\W+', '', orig_text).lower()
        clean_corr = re.sub(r'\W+', '', corr_text).lower()
        is_ocr = clean_orig == clean_corr

        # Get context window
        ctx_start = max(0, orig_start - 50)
        ctx_end = min(len(original), orig_end + 50)
        context = original[ctx_start:ctx_end]

        changes.append(TextChange(
            original=orig_text.strip(),
            corrected=corr_text.strip(),
            start=orig_start,
            end=orig_end,
            context=context.replace('\n', ' '),
            is_ocr_error=is_ocr
        ))

    return changes

def print_changes(changes: list[TextChange]):
    """Color-coded output for changes"""
    for change in changes:
        color = '\033[93m' if change.is_ocr_error else '\033[91m'
        reset = '\033[0m'

        print(f"{color}{change.original} → {change.corrected}{reset}")
        print(f"Position: {change.start}-{change.end}")
        print(f"Context: {change.context}\n")


In [ ]:
# Example usage
original_text = '''Y pues en la celestial Jeru- salen no ha mudado de condicion vuestra Benignidad, proseguid, o Niño tierno, y Dios Eterno, proseguid en bendecirles, y favo- recerles. Sean tan fervorosamen- te devotos de vuestra Admirable MADRE, que se porten como sus hijos, y hermanos de leche con Vos. Seran sabios, si fueren cas- tos; que no entra vuestra Sabi- duria, donde no ay mucha pure- za de conciencia. Crezcan en vuestro santo temor, y amor, co- como en los años, y mucho mas. Adelantense en la virtud, como en las letras, y mucho mas; has- ta que lleguen, por vuesetra imi- tacion, a ser varones perfectos, y consumados, agradables a vuestros ojos, y provechosos a la Republica, que libra casi to- da su felizidad en la acertada'''
corrected_text = """Y pues  en la  celestial Jeru-
salén no ha mudado de condición
vuestra  Benignidad,  proseguid
o  Niño  tierno, y  Dios Eterno,
proseguid en bendecirles, y favo-
recerles.  Sean tan fervorosamente
te devotos de vuestra  Admirable
MADRE, que le porten como sus
hijos, y hermanos de leche  con
Vos.  Serán sabios, si fueren cas-
tos ; que no entra vuestra  Sabi-
duría, donde no hay mucha pure-
za de  conciencia.   Crezcan en
vuestro santo temor, y amor, co-
mo en los años, y mucho  más.
Adelanten en la virtud,  como
en las letras, y mucho  más ; has-
ta que lleguen,  por vuestra imi-
tación, a ser varones perfectos,
y consumados,    agradables   a
vuestros ojos, y provechosos a
la República,  que confia-
da su felicidad en  la  acertada
cría:"""



changes = get_text_changes(original_text, corrected_text)
print_changes(changes)


salen → salén
Position: 29-34
Context: Y pues en la celestial Jeru- salen no ha mudado de condicion vuestra Benignidad, pro

condicion → condición
Position: 51-60
Context:  pues en la celestial Jeru- salen no ha mudado de condicion vuestra Benignidad, proseguid, o Niño tierno, y D

fervorosamen- → fervorosamente
Position: 175-188
Context: seguid en bendecirles, y favo- recerles. Sean tan fervorosamen- te devotos de vuestra Admirable MADRE, que se por

se → le
Position: 232-234
Context: samen- te devotos de vuestra Admirable MADRE, que se porten como sus hijos, y hermanos de leche con Vo

Seran → Serán
Position: 287-292
Context: rten como sus hijos, y hermanos de leche con Vos. Seran sabios, si fueren cas- tos; que no entra vuestra 

duria → duría
Position: 348-353
Context: s, si fueren cas- tos; que no entra vuestra Sabi- duria, donde no ay mucha pure- za de conciencia. Crezca

ay → hay
Position: 364-366
Context: - tos; que no entra vuestra Sabi- duria, donde no ay mucha pure- za de co

In [ ]:

# Save results to JSON
import json
with open("text_changes.json", "w") as f:
    json.dump([c.__dict__ for c in changes], f, indent=2)

In [ ]:
changes = get_text_changes(text2, corrected_text)
print_changes(changes)

salen → salén
Position: 29-34
Context: Y pues en la celestial Jeru- salen no ha mudado de condicion vuestra Benignidad, pro

condicion → condición
Position: 51-60
Context:  pues en la celestial Jeru- salen no ha mudado de condicion vuestra Benignidad, proseguid, o Niño tierno, y D

fervorosamen- → fervorosamente
Position: 175-188
Context: seguid en bendecirles, y favo- recerles. Sean tan fervorosamen- te devotos de vuestra Admirable MADRE, que se por

se → le
Position: 232-234
Context: samen- te devotos de vuestra Admirable MADRE, que se porten como sus hijos, y hermanos de leche con Vo

Seran → Serán
Position: 287-292
Context: rten como sus hijos, y hermanos de leche con Vos. Seran sabios, si fueren cas- tos; que no entra vuestra 

duria → duría
Position: 348-353
Context: s, si fueren cas- tos; que no entra vuestra Sabi- duria, donde no ay mucha pure- za de conciencia. Crezca

ay → hay
Position: 364-366
Context: - tos; que no entra vuestra Sabi- duria, donde no ay mucha pure- za de co

In [ ]:
import json
import re
from collections import defaultdict

# Input/Output files
TEXT_CHANGES_FILE = "text_changes.json"
SURFACE_FORMS_FILE = "surface_forms.json"
ORTHO_ERRORS_FILE = "orthographic_errors.json"

def normalize(text):
    """Normalize text for comparison (ignore accents and case)"""
    return re.sub(r'\W+', '', text).lower()

def categorize_changes():
    # Load text changes
    with open(TEXT_CHANGES_FILE, 'r') as f:
        changes = [TextChange(**c) for c in json.load(f)]

    # Initialize categories
    surface_forms = defaultdict(lambda: defaultdict(int))
    orthographic_errors = []

    for change in changes:
        orig = change.original
        corr = change.corrected

        # Surface form detection
        if change.is_ocr_error or normalize(orig) == normalize(corr):
            key = corr if change.is_ocr_error else normalize(corr)
            surface_forms[key][orig] += 1
        else:
            orthographic_errors.append({
                "original": orig,
                "corrected": corr,
                "position": (change.start, change.end),
                "context": change.context
            })

    # Save results
    with open(SURFACE_FORMS_FILE, 'w') as f:
        json.dump(surface_forms, f, indent=2, ensure_ascii=False)

    with open(ORTHO_ERRORS_FILE, 'w') as f:
        json.dump(orthographic_errors, f, indent=2, ensure_ascii=False)

    print(f"""
    Processing complete!
    - Surface forms: {len(surface_forms)} entries
    - Orthographic errors: {len(orthographic_errors)} entries
    """)

# Dataclass for JSON deserialization
from dataclasses import dataclass
@dataclass
class TextChange:
    original: str
    corrected: str
    start: int
    end: int
    context: str
    is_ocr_error: bool

if __name__ == "__main__":
    categorize_changes()


    Processing complete!
    - Surface forms: 0 entries
    - Orthographic errors: 16 entries
    
